# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset - note: `metadata` is an object (not a dict!)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Schema ID: {metadata.id if hasattr(metadata, 'id') else getattr(metadata, '@id', 'N/A')}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")


## 2. Data Overview
Review available record sets, their fields, and associated IDs.

To ensure clarity and consistency, all entities are referenced by their `@id` fields.

In [ ]:
# List all record sets and their fields

record_sets = list(dataset.record_sets)

if len(record_sets) == 0:
    print("No record sets found in this dataset's Croissant schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs.id if hasattr(rs, 'id') else getattr(rs, '@id', 'N/A')} (")
        print(f"  name: {rs.name if hasattr(rs, 'name') else '(no name)'}")
        print(f"  description: {rs.description if hasattr(rs, 'description') else '(no description)'}")
        print(f"  Fields:")
        for f in rs.fields:
            print(f"    - {f.id if hasattr(f, 'id') else getattr(f, '@id', 'N/A')}: {f.name if hasattr(f, 'name') else ''} ({f.data_type if hasattr(f, 'data_type') else ''})")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered in the overview above.

> **Note:** For this dataset, we'll attempt to programmatically extract the first available record set (if present) and show the fields.

In [ ]:
# Extract all records from all record sets into pandas DataFrames, indexed by their unique @id

dataframes = {}
record_set_ids = []

record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets available. Cannot extract records.")
else:
    for rs in record_sets:
        rs_id = rs.id if hasattr(rs, 'id') else getattr(rs, '@id', None)
        if not rs_id:
            continue
        record_set_ids.append(rs_id)
        # Extract records
        try:
            records = list(dataset.records(record_set=rs_id))
            if len(records) > 0:
                df = pd.DataFrame(records)
                dataframes[rs_id] = df
                print(f"Loaded {len(df)} records for RecordSet {rs_id}")
                print(f"Columns: {df.columns.tolist()}")
                display(df.head())
            else:
                print(f"RecordSet {rs_id} contains no records.")
        except Exception as e:
            print(f"Could not load records for RecordSet {rs_id}: {e}")

# Choose the main record set for further EDA (use the first one found)
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering numeric records, normalizing values, and grouping by key categorical fields, all using field `@id`s for references.

> **Tip:** You may need to tailor the field IDs below depending on the actual record set extracted.

In [ ]:
# EDA on the first numeric field (modify as needed)
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    numeric_fields = []
    for col in df.columns:
        # Try to infer if field is numeric
        sample_vals = df[col].dropna().head(10)
        try:
            # Try to parse as numbers
            sample_numeric = pd.to_numeric(sample_vals)
            numeric_fields.append(col)
        except Exception:
            continue

    if len(numeric_fields) == 0:
        print("No numeric fields detected in this record set.")
    else:
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field: {numeric_field}")
        # Try to parse as numeric for the full column
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanmean(df[numeric_field])  # For example, use the mean as a threshold
        filtered_df = df[df[numeric_field] > threshold]

        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by first categorical/text field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No categorical grouping field found.")
else:
    print("No usable data frame for EDA.")

## 5. Visualization
Visualize a histogram of the main numeric field distribution, and a boxplot grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and len(numeric_fields) > 0:
    df = dataframes[main_record_set_id]
    nf = numeric_fields[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[nf].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {nf}')
    plt.xlabel(nf)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=nf)
        plt.title(f'{nf} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(nf)
        plt.xlabel(group_field)
        plt.show()
else:
    print('No records or numeric fields to visualize.')

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to explore the FAIR² protein dataset with rigorous referencing by `@id`. We extracted summary statistics, explored numeric fields, visualized the distributions and identified group-level differences.

Further analysis could include more advanced feature engineering, missing data analysis, or machine learning applied to protein abundance or modification prediction.